<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson02/02_dense_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2: A dense neural network for images

1. Python basics & the training loop
2. **Dense neural network on FashionMNIST** ← *you are here*
3. Convolutional neural network (CNN) on FashionMNIST
4. The same with fastai
5. Fine-tuning a pretrained model

In notebook 1 we trained a line with 2 parameters. Now we train a real neural network with **~100,000 parameters** to recognize clothes in images. The good news: **the training loop stays exactly the same.**

**What you will learn here:**
- how images look to a computer (spoiler: just numbers)
- why we train in *batches*
- how to define a neural network as a class, and what is inside it
- how to turn the network's output into a prediction (*softmax*) and measure its error (*cross-entropy*)
- how to check where the model makes mistakes

**Before you start:** switch on the GPU via *Runtime → Change runtime type → T4 GPU*. (It also works on CPU, just slower.)

In [ ]:
import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

---
## 1. The data: FashionMNIST

[FashionMNIST](https://github.com/zalandoresearch/fashion-mnist) is a classic beginner dataset from Zalando: small grayscale photos of clothes in 10 categories.

The data comes in two parts:
- **training set** (60,000 images): the model learns from these
- **test set** (10,000 images): we check the model on these. They are like exam questions the model has never seen before.

`transforms.ToTensor()` converts each image into a tensor of numbers between 0 (black) and 1 (white).

In [ ]:
train_set = datasets.FashionMNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
test_set = datasets.FashionMNIST(root="data", train=False, download=True, transform=transforms.ToTensor())

print("training images:", len(train_set))
print("test images:    ", len(test_set))

In [ ]:
class_names = train_set.classes
class_names

### Look inside: one image

Each item of the dataset is a pair *(image, label)*. The label is a number from 0 to 9, an index into `class_names`.

In [ ]:
image, label = train_set[0]

print("image shape:", image.shape)
print("label:", label, "=", class_names[label])

plt.imshow(image[0], cmap="gray")
plt.title(class_names[label])
plt.show()

The shape `[1, 28, 28]` means **1 color channel** (grayscale; a color image would have 3: red, green, blue) and **28 × 28 pixels**.

To the computer, the image is just a grid of 784 numbers. Let's print them, scaled from 0-1 to 0-9 so each fits in one character (`.` = 0 = black):

In [ ]:
for row in image[0]:
    print(" ".join(str(round(p.item() * 9)) if p > 0 else "." for p in row))

Here are more examples:

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image, label = train_set[i]
    ax.imshow(image[0], cmap="gray")
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
plt.show()

---
## 2. Batches

In notebook 1, every training step used all 100 data points. With 60,000 images, that would be slow. Instead, each step uses a small random **batch** of images (here 64):

- compute the loss on 64 images → compute gradients → update the weights → next 64 images...
- after all 60,000 images are used once, one **epoch** is done
- this is why it is called **stochastic** gradient descent (SGD): each step sees a random sample of the data

A `DataLoader` cuts the dataset into batches for us. `shuffle=True` mixes the training images differently in every epoch.

In [ ]:
batch_size = 64

train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size)

print("batches (= training steps) per epoch:", len(train_loader))

### Look inside: one batch

`next(iter(train_loader))` simply means *"give me one batch"*:

In [ ]:
images, labels = next(iter(train_loader))

print("images:", images.shape)    # 64 images, 1 channel, 28x28 pixels
print("labels:", labels.shape)    # 64 labels
print("first 10 labels:", labels[:10])

---
## 3. The model

### Step 1: flatten the image

A dense layer expects a plain list of numbers, not a 2D grid. `nn.Flatten()` puts the 28 rows of pixels one after another into a single row of 784 numbers:

In [ ]:
flatten = nn.Flatten()
print(images.shape, "->", flatten(images).shape)

### Step 2: dense layers

Remember `nn.Linear(1, 1)` from notebook 1, which computed `weight * x + bias`? A **dense layer** is the same thing, just bigger:

- `nn.Linear(784, 128)` has **128 neurons**, and each neuron computes its own weighted sum of all 784 pixels (+ bias)
- "dense" (also "fully connected", hence `fc`) means that every neuron is connected to every input

Our network has three dense layers:

```
image 28×28  →  flatten: 784  →  dense: 128  →  dense: 64  →  dense: 10 (one score per class)
```

### Step 3: ReLU, the activation function

Between the layers we put **ReLU**: it keeps positive numbers and replaces negative ones with 0. It looks trivial, but it is essential. Without it, stacking linear layers would still give just one big linear function (a "line"), no matter how many layers we add.

In [ ]:
x = torch.linspace(-3, 3, 100)
plt.plot(x, torch.relu(x))
plt.title("ReLU(x) = max(0, x)")
plt.grid()
plt.show()

### Putting it together

Remember the class pattern from notebook 1: `__init__` lists the **layers**, `forward` describes how data **flows** through them. (`super().__init__()` is required boilerplate. Just always write it.)

In [ ]:
class DenseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)          # [batch, 1, 28, 28] -> [batch, 784]
        x = self.relu(self.fc1(x))   # -> [batch, 128]
        x = self.relu(self.fc2(x))   # -> [batch, 64]
        x = self.fc3(x)              # -> [batch, 10]   one score per class
        return x

model = DenseNet().to(device)        # create the network and move it to the GPU
model

### Look inside: how many parameters?

The same cell as in notebook 1, now for a real network:

In [ ]:
for name, param in model.named_parameters():
    print(f"{name:12s} shape = {str(list(param.shape)):12s} count = {param.numel():,}")

total = sum(p.numel() for p in model.parameters())
print(f"\ntotal number of parameters: {total:,}")

For example, `fc1.weight` has shape `[128, 784]`: 128 neurons, each with one weight per pixel. That alone is 100,352 numbers!

**All 109,386 parameters start random, and gradient descent will tune every one of them.** Our line from notebook 1 had 2.

### Look inside: follow one batch through the network

Let's push our batch of 64 images through the layers one by one and watch the shape change:

In [ ]:
x = images.to(device)
print("input:         ", x.shape)
x = model.flatten(x)
print("after flatten: ", x.shape)
x = model.relu(model.fc1(x))
print("after fc1+relu:", x.shape)
x = model.relu(model.fc2(x))
print("after fc2+relu:", x.shape)
x = model.fc3(x)
print("after fc3:     ", x.shape)

---
## 4. From scores to predictions: softmax

For each image, the network outputs **10 numbers**, one score per class. The class with the highest score is the prediction. The scores themselves can be any numbers (negative, large...), so we use **softmax** to turn them into **probabilities**: all between 0 and 1, summing to 1.

Let's see what the *untrained* network says about the first image of the batch:

In [ ]:
with torch.no_grad():                       # we are not training now, no need for gradients
    scores = model(images.to(device))

print("scores shape:", scores.shape)
print("scores for the 1st image:       ", scores[0])

probabilities = torch.softmax(scores, dim=1)
print("probabilities for the 1st image:", probabilities[0])
print("sum of probabilities:", probabilities[0].sum().item())

All probabilities are around 0.1 = 1/10: the untrained network is just guessing. Let's write a small helper to plot what the network thinks about an image. We will use it again after training.

In [ ]:
def show_prediction(model, image, label):
    model.eval()
    with torch.no_grad():
        scores = model(image.unsqueeze(0).to(device))      # unsqueeze: a "batch" of one image
        probabilities = torch.softmax(scores, dim=1)[0].cpu()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
    ax1.imshow(image[0], cmap="gray")
    ax1.set_title(f"true label: {class_names[label]}")
    ax1.axis("off")
    ax2.barh(class_names, probabilities)
    ax2.invert_yaxis()                                     # first class on top
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("predicted probability")
    plt.show()

image, label = test_set[0]
show_prediction(model, image, label)

### Measuring the error: cross-entropy loss

For classification, we don't use the mean squared error. We use **cross-entropy**. The idea is simple:

> **loss = −log(probability the model gave to the correct class)**

- correct class gets probability 1 → loss = 0 (perfect)
- correct class gets probability 0.1 → loss = 2.3 (random guessing among 10 classes)
- correct class gets probability close to 0 → loss is huge (confident and wrong: heavily punished)

A tiny example with 3 classes, where the correct class is class 0:

In [ ]:
loss_fn = nn.CrossEntropyLoss()      # note: it applies softmax internally, so the model outputs raw scores
correct_class = torch.tensor([0])

examples = {
    "confident & right": torch.tensor([[5.0, 0.0, 0.0]]),
    "unsure":            torch.tensor([[0.0, 0.0, 0.0]]),
    "confident & wrong": torch.tensor([[0.0, 5.0, 0.0]]),
}

for name, scores_example in examples.items():
    probs = torch.softmax(scores_example, dim=1)[0]
    loss = loss_fn(scores_example, correct_class)
    print(f"{name:18s} probabilities = {probs.numpy().round(2)}   loss = {loss.item():.3f}")

And the loss of our untrained network on the batch? It should be close to −log(0.1) ≈ 2.30:

In [ ]:
loss = loss_fn(scores, labels.to(device))
print("loss of the untrained network:", loss.item())

### Accuracy

The loss is what the model minimizes, but we humans prefer **accuracy**: what fraction of images is classified correctly? Let's write a function that computes it for a whole dataset:

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)          # index of the highest score
            correct += (predictions == labels).sum().item()
    return correct / len(loader.dataset)

print(f"test accuracy before training: {accuracy(model, test_loader):.1%}")

About 10%, a random guess. Time to train!

---
## 5. Training

Compare this with the training loop from notebook 1. **The five steps are identical.** The only additions:
- an extra loop: for each epoch, go through all batches
- `.to(device)` moves each batch to the GPU
- `model.train()` switches the model to training mode (it matters for some layers we'll meet in the next notebook)

One epoch = 938 batches = 938 gradient descent steps.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
n_epochs = 5

train_losses, train_accuracies, test_accuracies = [], [], []

for epoch in range(n_epochs):
    start = time.time()
    model.train()
    total_loss = 0

    for images, labels in train_loader:               # one batch of 64 images at a time
        images, labels = images.to(device), labels.to(device)

        predictions = model(images)                   # 1. FORWARD
        loss = loss_fn(predictions, labels)           # 2. LOSS
        optimizer.zero_grad()                         # 5. RESET
        loss.backward()                               # 3. BACKWARD
        optimizer.step()                              # 4. UPDATE

        total_loss += loss.item()

    train_losses.append(total_loss / len(train_loader))
    train_accuracies.append(accuracy(model, train_loader))
    test_accuracies.append(accuracy(model, test_loader))
    print(f"epoch {epoch + 1}/{n_epochs}:  loss = {train_losses[-1]:.3f},  "
          f"train accuracy = {train_accuracies[-1]:.1%},  test accuracy = {test_accuracies[-1]:.1%}  "
          f"({time.time() - start:.0f} s)")

In [ ]:
epochs = range(1, n_epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_losses, "o-")
ax1.set_xlabel("epoch")
ax1.set_title("training loss")

ax2.plot(epochs, train_accuracies, "o-", label="train")
ax2.plot(epochs, test_accuracies, "o-", label="test")
ax2.set_xlabel("epoch")
ax2.set_title("accuracy")
ax2.legend()
plt.show()

From 10% to around **85%** in a minute! (Your numbers will differ a bit: the weights start random and the batches are shuffled.) Note that the accuracy on the training images is a bit higher than on the test images. The model is slightly better on images it has already seen, a first hint of **overfitting**.

---
## 6. What did the model learn?

### Look inside: predictions

The same image as before training:

In [ ]:
image, label = test_set[0]
show_prediction(model, image, label)

**Try it:** Change the index `0` to other numbers (0 to 9999) and look at more predictions.

Let's look at many test images at once. Wrong predictions are shown in **red**:

In [ ]:
model.eval()
fig, axes = plt.subplots(3, 8, figsize=(13, 6))
for i, ax in enumerate(axes.flat):
    image, label = test_set[i]
    with torch.no_grad():
        predicted = model(image.unsqueeze(0).to(device)).argmax().item()
    ax.imshow(image[0], cmap="gray")
    ax.set_title(f"{class_names[predicted]}\n(true: {class_names[label]})", fontsize=8,
                 color="black" if predicted == label else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Which classes get confused?

A **confusion matrix** counts, for each true class (rows), how often the model predicted each class (columns). The diagonal = correct predictions.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

all_predictions, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        all_predictions.append(model(images.to(device)).argmax(dim=1).cpu())
        all_labels.append(labels)
all_predictions = torch.cat(all_predictions)
all_labels = torch.cat(all_labels)

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(all_labels, all_predictions, display_labels=class_names,
                                        xticks_rotation=45, colorbar=False, ax=ax)
plt.show()

Not surprisingly, the model mixes up *Shirt*, *T-shirt/top*, *Pullover* and *Coat*. At 28×28 pixels, those are hard even for humans!

### Look inside: what do the neurons look for?

Each of the 128 neurons in the first layer has 784 weights, one per pixel. We can reshape them back into a 28×28 image. **Red** pixels increase the neuron's output, **blue** pixels decrease it. In some neurons you can spot outlines of clothes (a collar, sleeves, the sole of a shoe), while others just look like noise. Dense networks are hard to interpret:

In [ ]:
weights = model.fc1.weight.detach().cpu()     # shape [128, 784]
limit = weights.abs().max()

fig, axes = plt.subplots(2, 8, figsize=(13, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(weights[i].reshape(28, 28), cmap="coolwarm", vmin=-limit, vmax=limit)
    ax.set_title(f"neuron {i}", fontsize=9)
    ax.axis("off")
plt.show()

---
## 7. Summary

| ingredient | notebook 1 | this notebook |
|---|---|---|
| **data** | 100 (hours, points) pairs | 60,000 images, served in batches of 64 |
| **model** | `nn.Linear(1, 1)`, 2 parameters | 3 dense layers + ReLU, 109,386 parameters |
| **loss** | mean squared error | cross-entropy |
| **optimizer** | SGD | SGD |
| **training loop** | forward → loss → backward → update → reset | **the same** (+ a loop over batches) |

The numbers you *choose* (learning rate, batch size, number of epochs, layer sizes) are called **hyperparameters**, to distinguish them from the *parameters* the model learns.

**A weakness of dense networks:** `Flatten` throws away the 2D structure of the image. The network doesn't know which pixels are neighbors, and a pattern learned in one corner of the image is useless in another corner. **Convolutional networks** fix this. That's the next notebook, and you will see that only the model definition changes!

### Exercises
1. Make the hidden layers bigger (e.g. 512 and 256) or smaller (e.g. 32 and 16). How does the number of parameters change? And the accuracy?
2. Train for 20 epochs instead of 5. Does test accuracy keep improving? Watch the gap between train and test accuracy.
3. Remove the ReLUs from `forward` (write `x = self.fc1(x)` instead of `x = self.relu(self.fc1(x))`, etc.). What happens to the accuracy?
4. Try learning rates `0.01` and `1.0`.
5. **Bonus:** Replace SGD with `torch.optim.Adam(model.parameters(), lr=0.001)`. Is it better?

(After changing the model, rerun the cells from `class DenseNet` onwards, so that you start from a fresh, untrained model.)